In [52]:
# ==================================================
# Imports
# ==================================================

import os
import sys
import json

import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from pathlib import Path
from folium.features import GeoJsonTooltip

In [54]:
# ==================================================
# Configuració projecte
# ==================================================

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.config import (
    DATA_PROCESSED,
)

from src.utils_io import (
    load_csv,
)

In [56]:
# ==================================================
# Crear directoris APP
# ==================================================

APP_DATA = Path("../app/data")
APP_MAPS = Path("../app/maps")

APP_DATA.mkdir(parents=True, exist_ok=True)
APP_MAPS.mkdir(parents=True, exist_ok=True)

In [58]:
# ==================================================
# Carregar datasets finals
# ==================================================

print("Carregant datasets finals...")

df_clusters = load_csv(
    DATA_PROCESSED / "cluster_dataset.csv"
)

df_geo = gpd.read_file(
    DATA_PROCESSED / "geospatial_dataset.geojson"
)

df_model = load_csv(
    DATA_PROCESSED / "modeling_full_timeseries.csv"
)

Carregant datasets finals...


DataSourceError: C:\Users\arera\Desktop\UOC\TFM\data\processed\geospatial_dataset.geojson: No such file or directory

In [ ]:
# ==================================================
# Exportar datasets per Streamlit
# ==================================================

print("Exportant datasets per Streamlit...")

df_clusters.to_csv(
    APP_DATA / "cluster_dataset.csv",
    index=False
)

df_model.to_csv(
    APP_DATA / "timeseries_dataset.csv",
    index=False
)

df_geo.to_file(
    APP_DATA / "geospatial_dataset.geojson",
    driver="GeoJSON"
)

In [36]:
# ==================================================
# Seleccionar últim any disponible
# ==================================================

ANY_FINAL = df_geo["any"].max()

df_last = df_geo[
    df_geo["any"] == ANY_FINAL
].copy()

In [38]:
# ==================================================
# Preparar variable cartogràfica
# ==================================================

VARIABLE = "indicador_gentrificacio"

df_last[VARIABLE] = pd.to_numeric(
    df_last[VARIABLE],
    errors="coerce"
)

vmin = df_last[VARIABLE].min()
vmax = df_last[VARIABLE].max()

In [40]:
# ==================================================
# Crear mapa Folium
# ==================================================

print("Generant mapa HTML final...")

m = folium.Map(
    location=[41.385, 2.17],
    zoom_start=12,
    tiles="CartoDB positron"
)

Generant mapa HTML final...


In [42]:
# ==================================================
# Funció colors
# ==================================================

def get_color(value):

    if pd.isna(value):
        return "#cccccc"

    if vmax == vmin:
        return "#999999"

    norm = (value - vmin) / (vmax - vmin)

    return mcolors.to_hex(
        plt.cm.RdYlBu_r(norm)
    )

In [44]:
# ==================================================
# GeoJSON layer
# ==================================================

folium.GeoJson(
    df_last,
    style_function=lambda x: {
        "fillColor": get_color(
            x["properties"][VARIABLE]
        ),
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.75,
    },
    tooltip=GeoJsonTooltip(
        fields=[
            "NOM",
            VARIABLE,
            "cluster"
        ],
        aliases=[
            "Barri",
            "Indicador",
            "Cluster"
        ],
        localize=True
    )
).add_to(m)

In [46]:
# ==================================================
# Llegenda
# ==================================================

legend_html = f"""
<div style="
    position: fixed;
    bottom: 30px;
    right: 30px;
    width: 220px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
">
    <b>Indicador de gentrificació</b><br>

    <div style="
        height: 20px;
        background: linear-gradient(
            to right,
            #4575b4,
            #91bfdb,
            #fc8d59,
            #d73027
        );
        margin-top: 8px;
        margin-bottom: 8px;
    "></div>

    <span style="float:left;">
        {vmin:.2f}
    </span>

    <span style="float:right;">
        {vmax:.2f}
    </span>
</div>
"""

m.get_root().html.add_child(
    folium.Element(legend_html)
)

In [48]:
# ==================================================
# Guardar mapa
# ==================================================

MAP_PATH = APP_MAPS / "mapa_gentrificacio.html"

m.save(MAP_PATH)

print("Mapa guardat:", MAP_PATH)

Mapa guardat: ..\app\maps\mapa_gentrificacio.html


In [50]:
# ==================================================
# Resum final
# ==================================================

print("\nExportació completada correctament.")
print(f"Any final exportat: {ANY_FINAL}")
print(f"Nombre de barris: {df_last['territori'].nunique()}")


Exportació completada correctament.
Any final exportat: 2025
Nombre de barris: 72
